# 📈 Project 5: Machine Learning System for Sales Forecasting
### RISE Internship — Machine Learning & AI | Tamizhan Skills

---

## 📌 Problem Statement
Businesses struggle to predict future sales accurately, leading to poor inventory planning and revenue loss. Traditional forecasting methods fail to capture complex trends and seasonality. Industry relies on machine learning models for accurate sales forecasting.

## 🎯 Objective
To build a machine learning model that predicts future sales using historical sales and business data.

## 🛠️ Tools
Python | NumPy | Pandas | Matplotlib | Seaborn | Scikit-learn | Jupyter Notebook

---

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    r2_score, mean_absolute_percentage_error
)
import os
os.makedirs('outputs', exist_ok=True)

print('✅ All libraries imported successfully!')

---
## Step 2: Historical Sales Dataset Ingestion
> We build 3 years of realistic daily sales data (2021–2023) with **trend**, **yearly seasonality**, **weekly cycles**, **holiday spikes**, and **random noise** — exactly how real retail sales data looks.

In [ ]:
np.random.seed(42)
dates = pd.date_range(start='2021-01-01', end='2023-12-31', freq='D')
n = len(dates)

# Sales components
trend       = np.linspace(500, 900, n)                         # upward business growth
seasonality = 150 * np.sin(2 * np.pi * np.arange(n) / 365)    # yearly cycle
weekly      = 80  * np.sin(2 * np.pi * np.arange(n) / 7)      # weekly pattern
noise       = np.random.normal(0, 40, n)                       # random variation

# Holiday boosts
holiday_boost = np.zeros(n)
for i, d in enumerate(dates):
    if (d.month == 12 and d.day >= 20): holiday_boost[i] = 300   # Christmas
    if (d.month == 11 and 1 <= d.day <= 10): holiday_boost[i] = 200  # Nov sale
    if (d.month == 1  and d.day <= 5):  holiday_boost[i] = 150   # New Year

sales = np.clip(trend + seasonality + weekly + noise + holiday_boost, 100, None)

df = pd.DataFrame({'Date': dates, 'Sales': sales.round(2)})
df['Year']      = df['Date'].dt.year
df['Month']     = df['Date'].dt.month
df['Day']       = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['DayOfYear'] = df['Date'].dt.dayofyear
df['WeekOfYear']= df['Date'].dt.isocalendar().week.astype(int)
df['Quarter']   = df['Date'].dt.quarter
df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)

print(f'Date range      : {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Total records   : {len(df)}')
print(f'Avg daily sales : ${df["Sales"].mean():.2f}')
print(f'Min / Max sales : ${df["Sales"].min():.2f} / ${df["Sales"].max():.2f}')
df.head(10)

---
## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
monthly = df.groupby(['Year','Month'])['Sales'].sum().reset_index()
monthly['YearMonth'] = pd.to_datetime(monthly[['Year','Month']].assign(Day=1))

fig, axes = plt.subplots(3, 1, figsize=(16, 12))
fig.suptitle('Sales Data — Exploratory Analysis', fontsize=14, fontweight='bold')

axes[0].plot(df['Date'], df['Sales'], color='#3498db', linewidth=0.6, alpha=0.7)
axes[0].plot(df['Date'], df['Sales'].rolling(30).mean(), color='#e74c3c', linewidth=2, label='30-day rolling avg')
axes[0].set_title('Daily Sales Over Time (2021–2023)')
axes[0].set_ylabel('Sales ($)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].bar(monthly['YearMonth'], monthly['Sales'],
            color=['#3498db','#2ecc71','#e74c3c'] * (len(monthly)//3 + 1), width=20, alpha=0.8)
axes[1].set_title('Monthly Total Sales')
axes[1].set_ylabel('Total Sales ($)')
axes[1].grid(alpha=0.3)

dow_avg = df.groupby('DayOfWeek')['Sales'].mean()
axes[2].bar(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], dow_avg.values,
            color='#9b59b6', alpha=0.8, edgecolor='black', linewidth=0.5)
axes[2].set_title('Average Sales by Day of Week')
axes[2].set_ylabel('Avg Sales ($)')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/01_eda_sales_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('EDA — Seasonality & Trend Analysis', fontsize=14, fontweight='bold')

month_avg = df.groupby('Month')['Sales'].mean()
axes[0,0].bar(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'],
              month_avg.values, color='#e67e22', alpha=0.85, edgecolor='black', linewidth=0.5)
axes[0,0].set_title('Average Sales by Month (Seasonality)')
axes[0,0].set_ylabel('Avg Sales ($)')
axes[0,0].grid(axis='y', alpha=0.3)

for year, color in zip([2021,2022,2023], ['#3498db','#2ecc71','#e74c3c']):
    y_data = df[df['Year']==year].groupby('DayOfYear')['Sales'].mean()
    axes[0,1].plot(y_data.index, y_data.values, label=str(year), color=color, linewidth=1.2, alpha=0.8)
axes[0,1].set_title('Year-over-Year Comparison')
axes[0,1].set_xlabel('Day of Year')
axes[0,1].legend()
axes[0,1].grid(alpha=0.3)

quarters = [df[df['Quarter']==q]['Sales'].values for q in [1,2,3,4]]
axes[1,0].boxplot(quarters, labels=['Q1','Q2','Q3','Q4'], patch_artist=True,
                  boxprops=dict(facecolor='#3498db', alpha=0.6),
                  medianprops=dict(color='red', linewidth=2))
axes[1,0].set_title('Sales Distribution by Quarter')
axes[1,0].set_ylabel('Sales ($)')
axes[1,0].grid(axis='y', alpha=0.3)

df['Rolling_Mean'] = df['Sales'].rolling(30).mean()
df['Rolling_Std']  = df['Sales'].rolling(30).std()
axes[1,1].plot(df['Date'], df['Rolling_Mean'], color='#2ecc71', linewidth=1.5, label='30d Mean')
axes[1,1].fill_between(df['Date'],
                        df['Rolling_Mean'] - df['Rolling_Std'],
                        df['Rolling_Mean'] + df['Rolling_Std'],
                        alpha=0.2, color='#2ecc71', label='±1 Std Dev')
axes[1,1].set_title('Rolling Mean & Std Deviation')
axes[1,1].legend()
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/02_seasonality_trend.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 4: Data Preprocessing & Feature Engineering
> **The key insight in time-series ML:** We can't use future data to predict the past. So we create **lag features** — using yesterday's sales, last week's sales, etc., as inputs to predict today. This is how real forecasting systems work.

In [ ]:
# Lag features
df['Lag_1']   = df['Sales'].shift(1)     # yesterday's sales
df['Lag_7']   = df['Sales'].shift(7)     # same day last week
df['Lag_30']  = df['Sales'].shift(30)    # same day last month
df['Lag_365'] = df['Sales'].shift(365)   # same day last year

# Rolling window features
df['Rolling_7_Mean']  = df['Sales'].shift(1).rolling(7).mean()
df['Rolling_30_Mean'] = df['Sales'].shift(1).rolling(30).mean()
df['Rolling_7_Std']   = df['Sales'].shift(1).rolling(7).std()

# Calendar flags
df['IsMonthStart'] = df['Date'].dt.is_month_start.astype(int)
df['IsMonthEnd']   = df['Date'].dt.is_month_end.astype(int)

# Holiday flag
df['IsHoliday'] = 0
df.loc[(df['Month']==12) & (df['Day']>=20), 'IsHoliday'] = 1
df.loc[(df['Month']==11) & (df['Day']<=10), 'IsHoliday'] = 1
df.loc[(df['Month']==1)  & (df['Day']<=5),  'IsHoliday'] = 1

# Drop rows with NaN (from lag creation)
df_model = df.dropna().copy()

feature_cols = [
    'Year','Month','Day','DayOfWeek','DayOfYear','WeekOfYear',
    'Quarter','IsWeekend','IsMonthStart','IsMonthEnd','IsHoliday',
    'Lag_1','Lag_7','Lag_30','Lag_365',
    'Rolling_7_Mean','Rolling_30_Mean','Rolling_7_Std'
]

X = df_model[feature_cols]
y = df_model['Sales']

print(f'Total features  : {len(feature_cols)}')
print(f'Usable rows     : {len(df_model)}')
print('\nFeature preview:')
X.head()

---
## Step 5: Trend & Seasonality Analysis

In [ ]:
corr = df_model[feature_cols + ['Sales']].corr()['Sales'].drop('Sales').sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#e74c3c' if v > 0 else '#3498db' for v in corr.values]
corr.plot(kind='barh', ax=ax, color=colors, edgecolor='black', linewidth=0.5)
ax.set_title('Feature Correlation with Sales', fontweight='bold')
ax.set_xlabel('Pearson Correlation Coefficient')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/03_feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 6: Time-Based Train / Test Split
> **Critical rule for time series:** Never use random splitting. If you randomly split, future data leaks into training — the model "cheats". Always split by date: train on earlier data, test on later data.

In [ ]:
split_date = '2023-07-01'
train_mask = df_model['Date'] < split_date
test_mask  = df_model['Date'] >= split_date

X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]
dates_test = df_model.loc[test_mask, 'Date']

# Scale
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {len(X_train)} days  ({df_model.loc[train_mask,"Date"].min().date()} → {df_model.loc[train_mask,"Date"].max().date()})')
print(f'Test : {len(X_test)} days   ({df_model.loc[test_mask,"Date"].min().date()} → {df_model.loc[test_mask,"Date"].max().date()})')

---
## Step 7: Machine Learning Model Training
> For forecasting (predicting a number, not a category) we use **regression** models. These are evaluated differently from classifiers — we measure how far off the predicted number is from the real number.

In [ ]:
models = {
    'Linear Regression' : LinearRegression(),
    'Ridge Regression'  : Ridge(alpha=1.0),
    'Decision Tree'     : DecisionTreeRegressor(max_depth=8, random_state=42),
    'Random Forest'     : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting' : GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    results[name] = {
        'model' : model,
        'y_pred': y_pred,
        'MAE'   : mean_absolute_error(y_test, y_pred),
        'RMSE'  : np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2'    : r2_score(y_test, y_pred),
        'MAPE'  : mean_absolute_percentage_error(y_test, y_pred) * 100,
    }
    print(f"  {name:22s}  MAE={results[name]['MAE']:.2f}  RMSE={results[name]['RMSE']:.2f}  R²={results[name]['R2']:.4f}  MAPE={results[name]['MAPE']:.2f}%")

---
## Step 8: Model Evaluation — Regression Metrics
> **What each metric means:**
> - **MAE** (Mean Absolute Error) — average dollar error. Easy to explain to business.
> - **RMSE** (Root Mean Squared Error) — penalises large errors more. Better for catching big misses.
> - **R²** — how much of sales variance the model explains. 1.0 = perfect, 0 = useless.
> - **MAPE** — error as a percentage. "Our forecast is off by X%" — easy to communicate.

In [ ]:
metrics_df = pd.DataFrame({
    name: [v['MAE'], v['RMSE'], v['R2'], v['MAPE']]
    for name, v in results.items()
}, index=['MAE','RMSE','R²','MAPE (%)']).T

print('=== Model Comparison ===')
print(metrics_df.round(4).to_string())

best_name = min(results, key=lambda k: results[k]['RMSE'])
best = results[best_name]
print(f'\n🏆 Best Model (lowest RMSE): {best_name}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('Model Evaluation — Regression Metrics', fontsize=14, fontweight='bold')

short_names = ['LR','Ridge','DT','RF','GB']
for ax, metric, color in zip(axes, ['MAE','RMSE','MAPE'], ['#3498db','#e74c3c','#f39c12']):
    vals = [results[n][metric] for n in results]
    bars = ax.bar(short_names, vals, color=color, alpha=0.85, edgecolor='black', linewidth=0.5)
    ax.set_title(f'{metric} — lower is better')
    ax.set_ylabel(metric)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                f'{val:.1f}', ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/04_model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 9: Forecast Visualization

In [ ]:
# Actual vs Predicted — best model
fig, axes = plt.subplots(2, 1, figsize=(16, 12))
fig.suptitle(f'Sales Forecast — {best_name}', fontsize=14, fontweight='bold')

axes[0].plot(df_model.loc[train_mask,'Date'], y_train, color='#3498db', linewidth=0.8, alpha=0.6, label='Train Actual')
axes[0].plot(dates_test, y_test, color='#2ecc71', linewidth=1.5, label='Test Actual')
axes[0].plot(dates_test, best['y_pred'], color='#e74c3c', linewidth=1.5, linestyle='--', label='Predicted')
axes[0].axvline(pd.to_datetime(split_date), color='black', linestyle=':', linewidth=1.5, label='Split')
axes[0].set_title('Full Timeline: Train | Test | Forecast')
axes[0].set_ylabel('Sales ($)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(dates_test, y_test, color='#2ecc71', linewidth=2, label='Actual Sales')
axes[1].plot(dates_test, best['y_pred'], color='#e74c3c', linewidth=2, linestyle='--', label='Predicted Sales')
axes[1].fill_between(dates_test, best['y_pred'] - best['RMSE'], best['y_pred'] + best['RMSE'],
                     alpha=0.15, color='#e74c3c', label='±RMSE band')
axes[1].set_title('Test Period: Actual vs Predicted (Zoomed)')
axes[1].set_ylabel('Sales ($)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/05_forecast_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# All models comparison
fig, ax = plt.subplots(figsize=(16, 7))
ax.plot(dates_test, y_test, color='black', linewidth=2.5, label='Actual', zorder=5)
for (name, res), color in zip(results.items(), ['#3498db','#9b59b6','#e67e22','#2ecc71','#e74c3c']):
    ax.plot(dates_test, res['y_pred'], linewidth=1.2, linestyle='--', alpha=0.8,
            color=color, label=f"{name} (RMSE={res['RMSE']:.1f})")
ax.set_title('All Models — Forecast Comparison', fontweight='bold')
ax.set_ylabel('Sales ($)')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/06_all_models_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Residual analysis
residuals = y_test.values - best['y_pred']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'Residual Analysis — {best_name}', fontsize=13, fontweight='bold')

axes[0].scatter(best['y_pred'], residuals, alpha=0.3, color='#3498db', s=10)
axes[0].axhline(0, color='red', linewidth=1.5, linestyle='--')
axes[0].set_title('Residuals vs Predicted')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residuals')
axes[0].grid(alpha=0.3)

axes[1].hist(residuals, bins=40, color='#9b59b6', alpha=0.8, edgecolor='black', linewidth=0.4)
axes[1].axvline(0, color='red', linewidth=2, linestyle='--')
axes[1].set_title('Residual Distribution')
axes[1].grid(alpha=0.3)

axes[2].plot(dates_test, residuals, color='#e74c3c', linewidth=0.8, alpha=0.7)
axes[2].axhline(0, color='black', linewidth=1.5, linestyle='--')
axes[2].fill_between(dates_test, residuals, 0, alpha=0.2, color='#e74c3c')
axes[2].set_title('Residuals Over Time')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/07_residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance (Random Forest)
rf_model = results['Random Forest']['model']
feat_imp = pd.Series(rf_model.feature_importances_, index=feature_cols).nlargest(12)

fig, ax = plt.subplots(figsize=(10, 6))
colors_imp = ['#e74c3c' if i < 5 else '#3498db' for i in range(len(feat_imp))]
feat_imp.sort_values().plot(kind='barh', ax=ax, color=colors_imp[::-1], edgecolor='black', linewidth=0.5)
ax.set_title('Feature Importance — Random Forest', fontweight='bold')
ax.set_xlabel('Importance Score')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/08_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 10: Business Insight Generation

In [ ]:
test_df = df_model[test_mask][['Date','Sales']].copy()
test_df['Predicted']  = best['y_pred']
test_df['Month_Year'] = test_df['Date'].dt.to_period('M')

monthly_summary = test_df.groupby('Month_Year').agg(
    Actual_Sales   =('Sales','sum'),
    Predicted_Sales=('Predicted','sum')
).reset_index()
monthly_summary['Error_%'] = (
    (monthly_summary['Actual_Sales'] - monthly_summary['Predicted_Sales'])
    / monthly_summary['Actual_Sales'] * 100
).round(2)

print('Monthly Forecast Accuracy:')
print(monthly_summary.to_string(index=False))

# Key business numbers
month_names_d = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
                 7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
dow_names = {0:'Monday',1:'Tuesday',2:'Wednesday',3:'Thursday',4:'Friday',5:'Saturday',6:'Sunday'}

print(f"\nKEY BUSINESS INSIGHTS:")
print(f"  Avg Monthly Revenue : ${df_model.groupby(['Year','Month'])['Sales'].sum().mean():,.2f}")
print(f"  Best Month          : {month_names_d[df_model.groupby('Month')['Sales'].mean().idxmax()]}")
print(f"  Worst Month         : {month_names_d[df_model.groupby('Month')['Sales'].mean().idxmin()]}")
print(f"  Best Day of Week    : {dow_names[df_model.groupby('DayOfWeek')['Sales'].mean().idxmax()]}")
print(f"  Weekend Sales Lift  : +${df_model[df_model['IsWeekend']==1]['Sales'].mean() - df_model[df_model['IsWeekend']==0]['Sales'].mean():.2f}")
print(f"  Holiday Sales Lift  : +${df_model[df_model['IsHoliday']==1]['Sales'].mean() - df_model[df_model['IsHoliday']==0]['Sales'].mean():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Business Insights — Sales Patterns', fontweight='bold', fontsize=13)

x = range(len(monthly_summary))
width = 0.35
axes[0].bar([i-width/2 for i in x], monthly_summary['Actual_Sales'],
            width, label='Actual', color='#2ecc71', alpha=0.85, edgecolor='black', linewidth=0.5)
axes[0].bar([i+width/2 for i in x], monthly_summary['Predicted_Sales'],
            width, label='Predicted', color='#e74c3c', alpha=0.85, edgecolor='black', linewidth=0.5)
axes[0].set_xticks(list(x))
axes[0].set_xticklabels([str(m) for m in monthly_summary['Month_Year']], rotation=45, ha='right')
axes[0].set_title('Monthly: Actual vs Predicted')
axes[0].set_ylabel('Total Sales ($)')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

yearly = df_model.groupby('Year')['Sales'].sum()
growth = yearly.pct_change() * 100
bars = axes[1].bar(yearly.index.astype(str), yearly.values,
                   color=['#3498db','#2ecc71','#e74c3c'], alpha=0.85, edgecolor='black', linewidth=0.5)
for i, (bar, yr) in enumerate(zip(bars, yearly.index)):
    if i > 0:
        g = growth.iloc[i]
        axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+500,
                     f'+{g:.1f}%', ha='center', fontsize=10, fontweight='bold', color='#27ae60')
axes[1].set_title('Year-over-Year Sales & Growth')
axes[1].set_ylabel('Total Sales ($)')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/09_business_insights.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save outputs
test_df.to_csv('outputs/sales_forecast_results.csv', index=False)
monthly_summary.to_csv('outputs/monthly_forecast_summary.csv', index=False)
print('✅ Saved: outputs/sales_forecast_results.csv')
print('✅ Saved: outputs/monthly_forecast_summary.csv')

---
## Step 11: Model Documentation & Final Summary

In [ ]:
print('=' * 60)
print('   FINAL MODEL SUMMARY')
print('=' * 60)
print(f'  Best Model : {best_name}')
print(f'  MAE        : {best["MAE"]:.2f}   (avg $ error per day)')
print(f'  RMSE       : {best["RMSE"]:.2f}   (penalises large errors)')
print(f'  R²         : {best["R2"]:.4f}  (variance explained)')
print(f'  MAPE       : {best["MAPE"]:.2f}%  (avg % error)')
print()
for f in sorted(os.listdir('outputs')):
    print(f'  ✅ {f}')
print('=' * 60)
print('   PROJECT 5 COMPLETE ✓')
print('=' * 60)

---
## 📊 Conclusion

| Metric | Value |
|--------|-------|
| Best Model | Ridge Regression |
| MAE | 42.49 |
| RMSE | 54.16 |
| R² | 0.8401 |
| MAPE | 5.39% |

---
*RISE Internship | Machine Learning & AI | Tamizhan Skills*